In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE


# ---------------------------------------------------------
# 1. SAMPLE IMBALANCED DATASET
# ---------------------------------------------------------

data = pd.DataFrame({
    "mileage_km": [
        25000, 30000, 35000, 40000, 45000,
        50000, 55000, 60000, 65000, 70000,
        75000, 80000, 85000, 90000, 95000,
        100000, 105000, 110000, 115000, 120000
    ],

    "service_gap_months": [
        3, 4, 5, 5, 6,
        7, 8, 8, 9, 10,
        11, 12, 13, 14, 15,
        16, 17, 18, 19, 20
    ],

    "engine_alerts": [
        0, 0, 1, 1, 1,
        2, 1, 2, 2, 3,
        2, 3, 3, 4, 4,
        5, 5, 6, 6, 7
    ],

    "brake_wear_percent": [
        20, 25, 28, 30, 35,
        40, 42, 45, 50, 55,
        58, 62, 65, 70, 75,
        80, 82, 85, 88, 92
    ],

    "failure": [
        0, 0, 0, 0, 0,
        0, 0, 0, 0, 0,
        0, 0, 0, 0, 0,
        0, 1, 1, 1, 1
    ]
})

print(data)

# ---------------------------------------------------------
# 2. SEPARATE INPUT AND TARGET
# ---------------------------------------------------------

X = data.drop(columns=["failure"])
y = data["failure"]

print("Original class distribution:")
print(y.value_counts())


# ---------------------------------------------------------
# 3. TRAIN-TEST SPLIT
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

print("\nTraining data before SMOTE:")
print(y_train.value_counts())

print("\nTesting data:")
print(y_test.value_counts())


# ---------------------------------------------------------
# 4. SCALE THE TRAINING DATA
# ---------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ---------------------------------------------------------
# 5. APPLY SMOTE ONLY TO TRAINING DATA
# ---------------------------------------------------------

smote = SMOTE(
    sampling_strategy="auto",
    k_neighbors=2,
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

print("\nTraining data after SMOTE:")
print(pd.Series(y_train_smote).value_counts())


# ---------------------------------------------------------
# 6. TRAIN THE MODEL
# ---------------------------------------------------------

model = LogisticRegression(max_iter=2000)

model.fit(X_train_smote, y_train_smote)


# ---------------------------------------------------------
# 7. MAKE PREDICTIONS
# ---------------------------------------------------------

predictions = model.predict(X_test_scaled)
probabilities = model.predict_proba(X_test_scaled)[:, 1]


# ---------------------------------------------------------
# 8. EVALUATE THE MODEL
# ---------------------------------------------------------

print("\nAccuracy:")
print(accuracy_score(y_test, predictions))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, predictions))

print("\nClassification report:")
print(classification_report(y_test, predictions))

print("\nROC-AUC:")
print(roc_auc_score(y_test, probabilities))

    mileage_km  service_gap_months  engine_alerts  brake_wear_percent  failure
0        25000                   3              0                  20        0
1        30000                   4              0                  25        0
2        35000                   5              1                  28        0
3        40000                   5              1                  30        0
4        45000                   6              1                  35        0
5        50000                   7              2                  40        0
6        55000                   8              1                  42        0
7        60000                   8              2                  45        0
8        65000                   9              2                  50        0
9        70000                  10              3                  55        0
10       75000                  11              2                  58        0
11       80000                  12              3   

In [3]:
print("\n1. ORIGINAL DATA VERIFICATION")
print("-" * 40)

class_counts = y.value_counts().sort_index()
class_percentages = y.value_counts(normalize=True).sort_index() * 100

print("Class counts:")
print(class_counts)

print("\nClass percentages:")
print(class_percentages.round(2))

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nImbalance ratio: {imbalance_ratio:.1f}:1")


1. ORIGINAL DATA VERIFICATION
----------------------------------------
Class counts:
failure
0    16
1     4
Name: count, dtype: int64

Class percentages:
failure
0    80.0
1    20.0
Name: proportion, dtype: float64

Imbalance ratio: 4.0:1
